# Example 1.1.1 / 2: Indexed Sets

Companion notebook for **1.1.1 Flat vs Indexed Sets** in the AMPL lab Documentation, section "Indexed Sets". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Indexed Sets

An indexed set declares one set per member of an index set. Here each product is sold in its own
market areas. From Python, an indexed set is assigned a dictionary of lists, one list per index.
`ampl.set["AREA"]["coils"]` is the one set `AREA["coils"]`, a `Set` object with the same methods
as a flat set:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
set AREA {PROD};          # AREA[p]: the areas where product p is sold

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.set["AREA"] = {"bands": ["east", "north"], "coils": ["east", "west", "export"]}

In [5]:
# run.py
print(ampl.set["AREA"]["bands"].to_list())
print(ampl.set["AREA"]["coils"].to_list())
print(ampl.set["AREA"]["coils"].size())

['east', 'north']
['east', 'west', 'export']
3


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. The data file gives each member of the family
with its subscript, `set AREA[bands] := ...;`:

In [6]:
%%writefile data.dat
set PROD := bands coils;
set AREA[bands] := east north;
set AREA[coils] := east west export;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display AREA;
display card(AREA["coils"]);

set AREA[bands] := east north;
set AREA[coils] := east west export;

card( AREA['coils']) = 3

